# Interactive Script: **Cloud Mask Data Cube**

**Author:** Baturalp Arisoy<br>
**Contact:** baturalp.arisoy@uni-wuerzburg.de - Call me Batu :)

## Overview
This notebook guides the user through the essentials of cloud mask data cube.

## Contents

1. [How to Generate Cloud Mask Data Cube](#1-how-to-generate-cloud-mask-data-cube)
2. [Generate Multiple Masks from Probability Map](#2-generate-multiple-masks-from-probability-map)
3. [Mask L2A Stack with Masking Layers](#3-mask-l2a-stack-with-masking-layers)
4. [Fully Automated Workflow](#4-fully-automated-workflow)
5. [Filter Data Cube by Cloud Percentage](#5-filter-data-cube-by-cloud-percentage)
6. [Update Existing Cloud Layers](#6-update-existing-cloud-layers)

---

> **TIP:**<br><br>
> If you want to proceed with co-registration, can directly skip to **Chapter 4** and generate cloud masked data cube. <br><br>
> However, the 1-3 chapters are good to understand the logic behind for further processes. 

## Import

In [3]:
from stac2cube import get_cloud_layers, mask_stac_clouds, mask_from_probability
import xarray as xr
import matplotlib.pyplot as plt
import numpy as np

> **Note:**<br>
> SKIP to Chapter 4 if you want to mask your generated initial data cube from Chapter 1. However, the first chapters explain the logic behind the workflow. 

## 1. How to Generate Cloud Mask Data Cube

> **Note:**<br>
> **get_cloud_layers** cannot return lazy array because s2cloudless algorithm must be applied to computed data. Therefore, use the following cells for long time series if you are on a machine with a strong computational power.<br>  Otherwise, use HPC Slurm job and read the exported NETCDF file.

### 1.1 Set the parameters

In [ ]:
polygon = "./polygons/test.gpkg"
daterange = ["2024-04-01", "2024-04-10"]
output = "./results/test_cloud.nc"
threshold = [40, 50, 60, 70, 80, 90]    # Percentages of desired binary masks. If None -> returns only cloud probability maps
clip_raster = False
masking = None

### 1.2 Get Cloud Layers as exported NETCDF

In [ ]:
cloud_stac = get_cloud_layers(polygon=polygon, daterange=daterange, output=output,
                                clip_raster=clip_raster,
                                threshold=threshold, masking=masking
                            )

### 1.3 Alternative: Read the file if you already exported before

In [13]:
cloud_path = "./results/test_cloud.nc"

with xr.open_dataset(cloud_path) as ds:
    cloud_stac = ds["Cloud_Stack"].load()

In [ ]:
cloud_stac

### 1.4 Select Cloud Layers

In [13]:
cloud_probability = cloud_stac.sel(band="cloud_prob")
cloud_mask = cloud_stac.sel(band="cloud_mask_40") # Example for 40% threshold, you can change it to any of the specified thresholds and check the different masks

### 1.5 Plot Probability Map

In [ ]:
cloud_prob = cloud_probability.sel(time="2024-04-08")

fig, ax = plt.subplots(figsize=(20, 15))
ax.axis('off')
ax.imshow(cloud_prob, cmap='gray')

### 1.6 Plot Binary Mask Map

In [ ]:
cloud_masking = cloud_mask.sel(time="2024-04-08")

fig, ax = plt.subplots(figsize=(20, 15))
ax.axis('off')
ax.imshow(cloud_masking, cmap='gray')

### 1.7 Compare with RGB

In [ ]:
# Select your initial data cube and time to visualize
path = "./results/test.nc"
time_to_visualize = '2024-04-08'

with xr.open_dataset(path) as ds:
    stac = ds["Spectral_Temporal_Stack"].load()


bands = ["red", "green", "blue"]

da_time = stac.sel(time=time_to_visualize)
da_rgb = da_time.sel(band=bands)
rgb_image = da_rgb.values.transpose(1, 2, 0)

p2 = np.percentile(rgb_image, 2)
p98 = np.percentile(rgb_image, 98)
rgb_clipped = np.clip(rgb_image, p2, p98)
rgb_normalized = (rgb_clipped - p2) / (p98 - p2)

plt.figure(figsize=(20, 15))
plt.imshow(rgb_normalized)
plt.title("Normalized RGB")
plt.axis("off")
plt.show()

## 2. Generate Multiple Masks from Probability Map

> **Note:**<br>
> In case threshold parameter was set to None in the previous chapter, you can create multiple masks in this chapter.

In [5]:
cloud_probability = cloud_stac.sel(band="cloud_prob")

In [64]:
# The list of threshold is the list of binary masks to be generated.
# You can add two more parameters to change the shape of clouds. Default: average_over=4, dilation_size=2. according to s2cloudless documentation:
# average_over: Size of the disk in pixels for performing convolution (averaging probability over pixels).
# dilation_size: Size of the disk in pixels for performing dilation. 
    
mask_stac = mask_from_probability(cloud_probability=cloud_probability, threshold=[40, 60, 70, 90]) # SET THRESHOLD VALUES HERE

In [ ]:
# Select date and mask band
mask_sliced = mask_stac.sel(time="2024-04-08")
mask_sliced = mask_sliced.sel(band="cloud_mask_90")

fig, ax = plt.subplots(figsize=(20, 15))
ax.axis('off')
ax.imshow(mask_sliced, cmap='gray')

In [66]:
new_band_names = set(map(str, mask_stac["band"].values))
base_bands = [b for b in map(str, cloud_stac["band"].values) if b not in new_band_names]
base = cloud_stac.sel(band=base_bands)
combined = xr.concat([base, mask_stac], dim="band").transpose("time", "band", "y", "x")
combined.name = "Cloud_Stack"

Optionaly export

In [ ]:
from stac2cube import export_stac
export_stac(combined, "./results/test_cloud.nc", overwrite=True, var_name="Cloud_Stack")

## 3. Mask L2A Stack with Masking Layers

> **Note:**
> This could be activated when running **get_cloud_layers**

### 3.1 Read Sentinel-2 L2A Netcdf and cloud file

In [2]:
# Initial Data Cube
with xr.open_dataset("./results/test.nc") as ds:
    stac = ds["Spectral_Temporal_Stack"].load()

# Cloud Mask Data Cube
with xr.open_dataset("./results/test_cloud.nc") as ds:
    cloud = ds["Cloud_Stack"].load()

In [ ]:
# Optional: Check if the dates are matching. If not, the function will fail.
def compare_times(da1, da2):
    times1 = set(da1.coords['time'].values)
    times2 = set(da2.coords['time'].values)
    missing_in_da2 = times1 - times2
    missing_in_da1 = times2 - times1
    
    if missing_in_da2:
        print("Times in first dataarray but not in second:")
        for t in sorted(missing_in_da2):
            print(t)
    else:
        print("No missing times in second dataarray from first.")
    
    if missing_in_da1:
        print("Times in second dataarray but not in first:")
        for t in sorted(missing_in_da1):
            print(t)
    else:
        print("No missing times in first dataarray from second.")

compare_times(stac, cloud)

### 3.2 Run *mask_stac_clouds*

> **Note:**
> At the moment, the user select only one threshold at once to mask the initial data cube.

In [ ]:
# Path to export masked stac layers
output = "./results/test_masked.nc"
# Select the mask layer (with percentage) to apply
mask_layer = "cloud_mask_70"

masked_stac = mask_stac_clouds(stac=stac, cloud=cloud, mask_layer= mask_layer, output=output)

# can also directly input netcdf paths
# mask_stac_clouds("./results/test.nc", "./results/test_cloud.nc", "./results/test_masked.nc")

In [ ]:
with xr.open_dataset("./results/test_masked.nc") as ds:
    masked_stac = ds["Spectral_Temporal_Stack"].load()

masked_stac

### 3.3 Plot Masked STAC

In [ ]:
time_to_visualize = '2024-04-08'
bands = ["red", "green", "blue"]

da_time = masked_stac.sel(time=time_to_visualize)
da_rgb = da_time.sel(band=bands)
rgb_image = da_rgb.values.transpose(1, 2, 0)

# Plot the image
plt.figure(figsize=(20, 15))
plt.imshow(rgb_image)
plt.title("Masked RGB")
plt.axis("off")
plt.show()

> **Information for test study area in entire 2024:**
- 71 items for query of get_stac_layers -> max_cc = 100
- 23 items for query of get_stac_layers -> max_cc = 10 
- 41 items for after using s2cloudless -> max_cloud = 10

Limiting max_cc parameter of get_stac_layers function disregards 18 meaningful scenes! Therefore, keep max_cc at 100% and apply s2cloudless masking to keep the max amount of meaningful scenes.

### 3.4 Cloud Percentage of the Scene

**Request cloud percentage of the scene**

In [ ]:
date = '2024-04-08'

masked_stac_sliced = masked_stac.sel(time=date)
cloud_percentage = masked_stac_sliced.cloud_percentage.values
print(f"Cloud Coverage Percentage on {date}: {cloud_percentage}%")

## 4. Fully Automated Workflow

> **Note:**
> - Below workflow calculates cloud layers for dates of the initial data cube.
> - Exports cloud layers dataset to desired path. 
> - Applies binary mask of the desired percentage.
> - Exports masked L2A dataset in the same folder. 

In [1]:
from stac2cube import get_cloud_layers

In [ ]:
masking = "./results/test.nc"                           # Path to image to be masked (the initial data cube).
threshold = 70                                          # Percentage of the desired binary masks. Can't be a list. Can be any number between 0-100.
output_masked= None                                     # Path to export masked data cube. If None -> The masked image will be exported in the same folder with extra prefix "_masked_<threshold>"
output_clouds = "./results/test_cloud.nc"               # Output of cloud probability layers, not masked image. IF None -> returns only masked data cube, but not cloud probability layers.

In [ ]:
get_cloud_layers(masking=masking, 
                    output_clouds=output_clouds, 
                    output_masked=output_masked,
                    threshold=threshold
                )

## 5. Filter Data Cube by Cloud Percentage

> **Note:**<br><br>
> Unlike stac query of max_cc (entire Sentinel-2 tile), following cloud percentage represents the cloud coverage of the actual scene.

**Select scenes with less than max cloud coverage percentage**

In [ ]:
import xarray as xr

with xr.open_dataset("./results/test_masked70.nc") as ds:
    masked_stac = ds["Spectral_Temporal_Stack"].load()

masked_stac

In [ ]:
masked_stac.cloud_percentage.mean()

In [ ]:
from stac2cube import cloud_filter

max_cloud = 5
masked_stac_filtered = cloud_filter(masked_stac, max_cloud = max_cloud)
masked_stac_filtered

In [ ]:
masked_stac_filtered.cloud_percentage.mean()

**Optionally, can export the filtered stac**

In [ ]:
from stac2cube import export_stac

export_stac(masked_stac_filtered, "./results/test_cloud_filtered.nc")

## 6. Update Existing Cloud Layers

> **Warning:**
> Always run one of the options (5.1 or 5.2) below, not both of them in a single run! Updating mechanism alters the original files and causes problem when the other option is executed.<br> <br>
> Once the updating mechanism is triggered, the other option won't be effective until new dates are ingested.

In [1]:
import xarray as xr
from stac2cube import get_cloud_layers

### 6.1 Update Only Cloud Layers

In [ ]:
with xr.open_dataset("./results/test_cloud.nc") as ds:
    stac = ds["Cloud_Stack"].load()

stac.time.values

In [5]:
update = "./results/test_cloud.nc"
daterange = ["2024-04-04", "2024-04-20"]
threshold = None    # Percentages of desired binary masks. If None -> returns only cloud probability maps
clip_raster = False # does not work for now
output=None         # set it to a path if you want to export the cloud probability layers as a netcdf file

In [ ]:
cloud_stac = get_cloud_layers(update=update, daterange=daterange,
                                clip_raster=clip_raster, threshold=threshold,
                                output=output
                            )

In [ ]:
cloud_stac

In [ ]:
cloud_stac.time.values

### 6.2 Update both Cloud Layers and Masked L2A

> **Note:** <br><br>
> Initial Data Cube should have been already updated at **Chapter 5 in 1_Initial_Data_Cube.ipynb**<br><br>
> To update every data cube at the same time, please refer to **Chapter 3 in 5_Batch_Processing.ipynb** (under development!)

In [ ]:
stac_before_path = "./results/test_masked70.nc"

with xr.open_dataset(stac_before_path) as ds:
    stac_before = ds["Spectral_Temporal_Stack"].load()

stac_before.time.values

In [4]:
update = "./results/test_cloud.nc"
masking = "./results/test.nc"
threshold = 70                        # Percentage of the desired binary masks. Can't be a list. Can be any number between 0-100.

In [ ]:
updated_mask= get_cloud_layers(update=update, masking=masking, threshold=threshold)

In [ ]:
stac_after_path = "./results/test_masked70.nc"

with xr.open_dataset(stac_after_path) as ds:
    stac_after = ds["Spectral_Temporal_Stack"].load()

stac_after.time.values